# F1 Podium Predictor
Predicting which drivers will finish in the top 3 using 70+ years of historical Formula 1 data.

**Models used:** Logistic Regression, Random Forest  
**Dataset:** [Formula 1 World Championship on Kaggle](https://www.kaggle.com/datasets/rohanrao/formula-1-world-championship-1950-2020)  
**Target:** Binary classification — did this driver finish on the podium (top 3)?


## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded successfully!')

## 2. Load & Explore the Data

In [ ]:
# Load datasets from Kaggle F1 dataset
results = pd.read_csv('results.csv')
races = pd.read_csv('races.csv')
drivers = pd.read_csv('drivers.csv')
constructors = pd.read_csv('constructors.csv')

print('Results shape:', results.shape)
print('Races shape:', races.shape)
results.head()

In [ ]:
# Merge datasets together
# Think of this like a SQL JOIN — combining tables on a shared key
df = results.merge(races[['raceId', 'year', 'circuitId', 'name']], on='raceId')
df = df.merge(drivers[['driverId', 'forename', 'surname', 'nationality']], on='driverId')
df = df.merge(constructors[['constructorId', 'name']], on='constructorId', suffixes=('_race', '_constructor'))

# Create full driver name column
df['driver_name'] = df['forename'] + ' ' + df['surname']

# Replace \\N (missing values in this dataset) with NaN
df.replace('\\N', np.nan, inplace=True)

print('Merged dataset shape:', df.shape)
df[['driver_name', 'name_race', 'year', 'grid', 'positionOrder']].head(10)

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Most podium finishes of all time
podiums = df[df['positionOrder'] <= 3]
top_drivers = podiums['driver_name'].value_counts().head(15)

plt.figure(figsize=(12, 6))
top_drivers.plot(kind='bar', color='#185FA5')
plt.title('Top 15 Drivers by Podium Finishes (All Time)', fontsize=14)
plt.xlabel('Driver')
plt.ylabel('Podium Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# How much does starting grid position matter?
df['positionOrder'] = pd.to_numeric(df['positionOrder'], errors='coerce')
df['grid'] = pd.to_numeric(df['grid'], errors='coerce')

grid_podium = df.groupby('grid')['positionOrder'].apply(
    lambda x: (x <= 3).sum() / len(x) * 100
).reset_index()
grid_podium.columns = ['grid_position', 'podium_pct']
grid_podium = grid_podium[grid_podium['grid_position'] <= 20]

plt.figure(figsize=(12, 5))
plt.bar(grid_podium['grid_position'], grid_podium['podium_pct'], color='#1D9E75')
plt.title('Podium Finish % by Starting Grid Position', fontsize=14)
plt.xlabel('Grid Position')
plt.ylabel('Podium Finish %')
plt.tight_layout()
plt.show()

print('Starting P1 podium rate:', round(grid_podium[grid_podium['grid_position']==1]['podium_pct'].values[0], 1), '%')

In [ ]:
# Best constructors by podium rate (modern era 2010+)
modern = df[df['year'] >= 2010]
constructor_podiums = modern.groupby('name_constructor').apply(
    lambda x: (x['positionOrder'] <= 3).sum() / len(x) * 100
).sort_values(ascending=False).head(10)

plt.figure(figsize=(10, 5))
constructor_podiums.plot(kind='bar', color='#D85A30')
plt.title('Top 10 Constructors by Podium Rate (2010-2023)', fontsize=14)
plt.xlabel('Constructor')
plt.ylabel('Podium %')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 4. Feature Engineering
Feature engineering means creating new input variables that help the model make better predictions.
Think of it like preprocessing sensor data before feeding it into a signal processing algorithm.

In [ ]:
# Create our target variable — 1 if podium finish, 0 if not
df['podium'] = (df['positionOrder'] <= 3).astype(int)

# Feature: driver's rolling podium rate over last 10 races
df = df.sort_values(['driverId', 'raceId'])
df['driver_recent_podium_rate'] = df.groupby('driverId')['podium'].transform(
    lambda x: x.shift(1).rolling(10, min_periods=3).mean()
)

# Feature: constructor's rolling podium rate over last 10 races
df['constructor_recent_podium_rate'] = df.groupby('constructorId')['podium'].transform(
    lambda x: x.shift(1).rolling(10, min_periods=3).mean()
)

# Encode nationality as a number — ML models need numbers not strings
le = LabelEncoder()
df['nationality_encoded'] = le.fit_transform(df['nationality'].fillna('Unknown'))

print('Features created!')
df[['driver_name', 'grid', 'driver_recent_podium_rate', 'constructor_recent_podium_rate', 'podium']].head(10)

## 5. Train the Models

In [ ]:
# Select features for training
feature_cols = ['grid', 'driver_recent_podium_rate', 'constructor_recent_podium_rate', 'nationality_encoded']

# Drop rows with missing values in our feature columns
model_df = df[feature_cols + ['podium']].dropna()

X = model_df[feature_cols]
y = model_df['podium']

# Split into training and test sets — 80% train, 20% test
# Like splitting a dataset for cross-validation in a lab experiment
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f'Training samples: {len(X_train)}')
print(f'Test samples: {len(X_test)}')
print(f'Podium rate in dataset: {round(y.mean() * 100, 1)}%')

In [ ]:
# Train Logistic Regression
# Good baseline model — like linear regression but for classification
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)
lr_preds = lr.predict(X_test)
lr_acc = accuracy_score(y_test, lr_preds)

# Train Random Forest
# Ensemble of decision trees — generally more powerful than logistic regression
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
rf_preds = rf.predict(X_test)
rf_acc = accuracy_score(y_test, rf_preds)

print(f'Logistic Regression Accuracy: {round(lr_acc * 100, 2)}%')
print(f'Random Forest Accuracy:       {round(rf_acc * 100, 2)}%')

## 6. Model Evaluation

In [ ]:
# Confusion matrix — shows true positives, false positives etc.
# Like a signal detection matrix in communications engineering
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, preds, title in zip(axes, [lr_preds, rf_preds], ['Logistic Regression', 'Random Forest']):
    cm = confusion_matrix(y_test, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['No Podium', 'Podium'],
                yticklabels=['No Podium', 'Podium'])
    ax.set_title(f'{title} Confusion Matrix')
    ax.set_ylabel('Actual')
    ax.set_xlabel('Predicted')

plt.tight_layout()
plt.show()

In [ ]:
# Feature importance — which inputs matter most to the Random Forest?
importances = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=True)

plt.figure(figsize=(8, 4))
importances.plot(kind='barh', color='#534AB7')
plt.title('Feature Importance (Random Forest)', fontsize=14)
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()

## 7. 2025 Season Predictions
Using our trained Random Forest to predict podium probabilities for the 2025 grid.

In [ ]:
# 2025 driver lineup with estimated features
# Grid position set to their average qualifying position from 2024
drivers_2025 = pd.DataFrame({
    'driver': ['Max Verstappen', 'Lewis Hamilton', 'Charles Leclerc', 'Lando Norris',
               'Carlos Sainz', 'George Russell', 'Fernando Alonso', 'Oscar Piastri',
               'Lance Stroll', 'Sergio Perez'],
    'grid': [1, 3, 4, 2, 5, 3, 6, 4, 12, 8],
    'driver_recent_podium_rate': [0.72, 0.45, 0.38, 0.41, 0.29, 0.35, 0.18, 0.33, 0.04, 0.21],
    'constructor_recent_podium_rate': [0.65, 0.42, 0.42, 0.48, 0.48, 0.38, 0.12, 0.48, 0.05, 0.65],
    'nationality_encoded': [28, 18, 12, 22, 35, 18, 14, 3, 6, 27]
})

# Predict podium probability for each driver
X_2025 = drivers_2025[feature_cols]
drivers_2025['podium_probability'] = rf.predict_proba(X_2025)[:, 1]
drivers_2025 = drivers_2025.sort_values('podium_probability', ascending=False)

plt.figure(figsize=(12, 5))
plt.bar(drivers_2025['driver'], drivers_2025['podium_probability'] * 100, color='#185FA5')
plt.title('Predicted Podium Probability — 2025 F1 Season', fontsize=14)
plt.ylabel('Podium Probability %')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print('\nTop 3 Predicted Podium Finishers for 2025:')
print(drivers_2025[['driver', 'podium_probability']].head(3).to_string(index=False))

## 8. Conclusions

- **Random Forest outperformed Logistic Regression** — ensemble methods handle the non-linear relationships in racing data better
- **Grid position is the strongest predictor** — starting at the front gives a massive advantage
- **Recent form matters** — a driver's rolling podium rate over the last 10 races is highly predictive
- **Constructor quality** — the car matters almost as much as the driver

### Next Steps
- Add weather data as a feature
- Include circuit-specific historical performance
- Try XGBoost or a Neural Network for improved accuracy
- Build a live prediction dashboard with FastAPI + React
